In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Pagtukoy ng mga Opsyon ng Sampler

<Accordion>
<AccordionItem title="Mga bersyon ng pakete">

Ang code sa page na ito ay binuo gamit ang mga sumusunod na kinakailangan.
Inirerekomenda naming gamitin ang mga bersyong ito o mas bago.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
Maaari mong gamitin ang mga opsyon upang i-customize ang Sampler primitive. Ang seksyong ito ay nakatuon sa kung paano tukuyin ang mga opsyon ng Qiskit Runtime primitive. Bagaman ang interface ng `run()` na pamamaraan ng mga primitive ay pare-pareho sa lahat ng implementasyon, hindi ganoon ang mga opsyon. Tingnan ang kaukulang mga reperensya ng API para sa impormasyon tungkol sa mga opsyon ng [`qiskit.primitives.BackendSamplerV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BackendSamplerV2) at [`qiskit_aer.primitives.SamplerV2`](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.primitives.SamplerV2.html).

<span id="pass-options"></span>
## Itakda ang mga opsyon ng Sampler
Maaari kang magtakda ng mga opsyon kapag sinisimulan ang Sampler, pagkatapos masimulan ang Sampler, o maaari mong i-update ang mga opsyon pagkatapos masimulan ang Sampler. Para sa mga tagubilin sa paggamit ng mga pamamaraang ito, tingnan ang paksa na [Panimula sa mga opsyon](/guides/runtime-options-overview#options-precedence).

Bukod dito, maaari mong itakda ang halaga ng `shots` sa pamamaraang `run()`, ayon sa inilalarawan sa susunod na seksyon.
<span id="run-method"></span>
### Pamamaraan ng Run()
Ang mga halagang maaari mong ipasa sa `run()` ay ang mga tinukoy sa interface lamang, iyon ay ang `shots`. Isinusulat nito ang anumang halagang itinakda para sa `default_shots` para sa kasalukuyang run.

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit1 = random_iqp(3)
circuit1.measure_all()
circuit2 = random_iqp(3)
circuit2.measure_all()

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

transpiled1 = pass_manager.run(circuit1)
transpiled2 = pass_manager.run(circuit2)

sampler = Sampler(mode=backend)
# Default shots to use if not specified in run()
sampler.options.default_shots = 500
# Sample two circuits at 128 shots each.
sampler.run([transpiled1, transpiled2], shots=128)

<RuntimeJobV2('d8286680bvlc73d1vmu0', 'sampler')>

### Special cases

<span id="shots"></span>
#### Shots

The `SamplerV2.run` method accepts two arguments: a list of PUBs, each of which can specify a PUB-specific value for shots, and a shots keyword argument. These shot values are a part of the Sampler execution interface, and are independent of the Runtime Sampler's options.  They take precedence over any values specified as options in order to comply with the Sampler abstraction.

However, if `shots` is not specified by any PUB or in the run keyword argument (or if they are all `None`), then the shots value from the options is used, most notably `default_shots`.

To summarize, this is the order of precedence for specifying shots in the Sampler, for any particular PUB:

1. If the PUB specifies shots, use that value.
2. If the `shots` keyword argument is specified in `run`, use that value.
4. If `twirling` is enabled  (True by default), then the product of `num_randomizations` and `shots_per_randomization`, as specified as  [`twirling` options](/docs/api/qiskit-ibm-runtime/options-twirling-options), is used.
5. If `sampler.options.default_shots` is specified, use that value.

Thus, if shots are specified in all possible places, the one with highest precedence (shots specified in the PUB) is used.

Example:

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit1 = random_iqp(3)
circuit1.measure_all()
circuit2 = random_iqp(3)
circuit2.measure_all()

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

transpiled1 = pass_manager.run(circuit1)
transpiled2 = pass_manager.run(circuit2)


# Setting shots during primitive initialization
sampler = Sampler(mode=backend, options={"default_shots": 4096})

# Setting options after primitive initialization
# This uses auto-complete.
sampler.options.default_shots = 2000

# This does bulk update.  The value for default_shots is overridden
# if you specify shots with run() or in the PUB.
sampler.options.update(
    default_shots=1024, dynamical_decoupling={"sequence_type": "XpXm"}
)

# Sample two circuits at 128 shots each.
sampler.run([transpiled1, transpiled2], shots=128)

<RuntimeJobV2('d82868ugbeec73alfa80', 'sampler')>

### Mga espesyal na kaso
<span id="shots"></span>
#### Shots
Ang pamamaraang `SamplerV2.run` ay tumatanggap ng dalawang argumento: isang listahan ng mga PUB, kung saan ang bawat isa ay maaaring tumukoy ng partikular na halaga ng shots para sa PUB, at isang keyword na argumento ng shots. Ang mga halagang shots na ito ay bahagi ng interface ng pagpapatakbo ng Sampler, at hiwalay sa mga opsyon ng Runtime Sampler. Mas mataas ang kanilang priyoridad kaysa sa anumang halagang itinukoy bilang mga opsyon upang sumunod sa abstraction ng Sampler.

Gayunpaman, kung ang `shots` ay hindi tinukoy ng anumang PUB o sa run keyword na argumento (o kung lahat ay `None`), ang halaga ng shots mula sa mga opsyon ang gagamitin, lalo na ang `default_shots`.

Upang ibuod, ito ang pagkakasunod-sunod ng priyoridad para sa pagtukoy ng shots sa Sampler, para sa anumang partikular na PUB:

1. Kung ang PUB ay tumutukoy ng shots, gamitin ang halagang iyon.
2. Kung ang keyword na argumento ng `shots` ay tinukoy sa `run`, gamitin ang halagang iyon.
4. Kung ang `twirling` ay pinagana (True bilang default), ang produkto ng `num_randomizations` at `shots_per_randomization`, ayon sa tinukoy bilang [mga opsyon ng `twirling`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options), ay gagamitin.
5. Kung ang `sampler.options.default_shots` ay tinukoy, gamitin ang halagang iyon.

Kaya, kung ang shots ay tinukoy sa lahat ng posibleng lugar, ang isa na may pinakamataas na priyoridad (shots na tinukoy sa PUB) ang gagamitin.

Halimbawa: